# 02 — Train Tier 1: QLoRA SFT

**Runs on:** Google Colab Free (T4 15 GB).

## How to use this notebook
1. Open in Colab. Set runtime → GPU (T4).
2. (Optional) Edit the `MODEL_NAME` in the **CONFIG** cell to the winner from `02a_model_evaluation.ipynb`. Default is Phi-4-mini.
3. **Runtime → Run all.**
4. When prompted, upload `train.jsonl`, `val.jsonl`, `test.jsonl` (from `tools/yen_sei/data/refined/`).
5. Wait ~3–6 hours. The notebook will:
   - QLoRA fine-tune the chosen model (3 epochs, early stop on val loss).
   - Run inference on the test split + report JSON-compliance and quality metrics.
   - Save adapter, merged-fp16 weights (if VRAM allows), and zip them for download.

**Total user effort: pick the runtime, click Run all, drop three files. No copy-paste of code.**


In [ ]:
# =====================================================================
# Cell 1 — Install dependencies (silent). ~2-3 min on first run.
# =====================================================================
import subprocess, sys, os, time

PKGS = [
    "unsloth",
    "peft>=0.11.0",
    "transformers>=4.45.0",
    "datasets>=2.20.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.34.0",
    "trl>=0.10.0",
]
print("Installing dependencies (silent)…")
t0 = time.time()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS],
    check=True,
)
print(f"Done in {time.time() - t0:.1f}s.")


In [ ]:
# =====================================================================
# Cell 2 — CONFIG. Edit MODEL_NAME if 02a picked Gemma instead of Phi.
# =====================================================================

# Default winner; override after running 02a.
MODEL_NAME  = "microsoft/Phi-4-mini-instruct"

# Hyperparameters from PLAN.md
MAX_SEQ_LEN  = 2048
EPOCHS       = 3
BATCH_SIZE   = 4
GRAD_ACCUM   = 4         # effective batch = 16
LR           = 2e-4
LORA_R       = 32
LORA_ALPHA   = 64
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

# Limits (set to None to use all). Useful for quick smoke runs.
TRAIN_LIMIT  = None      # full ~2,859 train rows
VAL_LIMIT    = None      # full ~227 val rows
TEST_LIMIT   = None      # full ~231 test rows

# Output / save behaviour
OUT_DIR             = "/content/yen_sei_tier1"
SAVE_MERGED_FP16    = True   # merge LoRA into base + save fp16 (large; download as zip)
PUSH_TO_HF          = False  # set True + add HF_TOKEN below to push adapter to your hub
HF_REPO_ID          = ""     # e.g. "your-username/yen-sei-tier1-phi4-lora"
HF_TOKEN            = ""     # paste a write token if PUSH_TO_HF=True

os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"OUT_DIR    = {OUT_DIR}")


## Upload data

The next cell looks for `train.jsonl`, `val.jsonl`, `test.jsonl` in `/content/`. If missing, it opens an upload widget — drop all three files from `tools/yen_sei/data/refined/`.


In [ ]:
# =====================================================================
# Cell 3 — Locate (or upload) train/val/test JSONL files.
# =====================================================================
import json
from pathlib import Path

REQUIRED = ["train.jsonl", "val.jsonl", "test.jsonl"]
SEARCH_DIRS = [Path("/content"), Path.cwd(), Path("/content/drive/MyDrive/yen_sei")]

def find_data():
    found = {}
    for name in REQUIRED:
        for d in SEARCH_DIRS:
            p = d / name
            if p.exists():
                found[name] = p
                break
    return found

found = find_data()
missing = [n for n in REQUIRED if n not in found]

if missing:
    print(f"Missing: {missing}. Opening upload widget…")
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        for name in uploaded:
            dest = Path("/content") / name
            print(f"  saved {name} -> {dest} ({len(uploaded[name])} bytes)")
        found = find_data()
    except ImportError:
        raise SystemExit(
            "Not in Colab. Place train.jsonl, val.jsonl, test.jsonl next to this "
            "notebook or in /content/."
        )

assert all(n in found for n in REQUIRED), f"Still missing: {set(REQUIRED) - set(found)}"
TRAIN_PATH = found["train.jsonl"]
VAL_PATH   = found["val.jsonl"]
TEST_PATH  = found["test.jsonl"]
print(f"\ntrain: {TRAIN_PATH}\nval:   {VAL_PATH}\ntest:  {TEST_PATH}")

def load_jsonl(path, limit=None):
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows

train_rows = load_jsonl(TRAIN_PATH, limit=TRAIN_LIMIT)
val_rows   = load_jsonl(VAL_PATH,   limit=VAL_LIMIT)
test_rows  = load_jsonl(TEST_PATH,  limit=TEST_LIMIT)
print(f"\nLoaded train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}.")


## Train (QLoRA SFT)

Loads the model in 4-bit, attaches LoRA adapters, runs SFT for `EPOCHS` epochs with cosine LR schedule and warmup. Saves the best checkpoint by val loss. ~3-6 hours on T4 for the full dataset.


In [ ]:
# =====================================================================
# Cell 4 — Load model + LoRA adapter, prepare dataset, train.
# =====================================================================
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

print(f"Loading {MODEL_NAME} in 4-bit…")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )}

train_ds = Dataset.from_list(train_rows).map(to_text, remove_columns=["messages"])
val_ds   = Dataset.from_list(val_rows).map(to_text,   remove_columns=["messages"])

steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_RATIO))
print(f"\nTotal steps: {total_steps}  (steps/epoch={steps_per_epoch}, warmup={warmup_steps})")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=SFTConfig(
        output_dir=os.path.join(OUT_DIR, "checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        weight_decay=WEIGHT_DECAY,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        report_to="none",
        seed=42,
    ),
)

t0 = time.time()
trainer.train()
train_secs = time.time() - t0
peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0
print(f"\nTraining done in {train_secs/60:.1f} min, peak VRAM: {peak_vram:.1f} GB")


## Evaluate on test set


In [ ]:
# =====================================================================
# Cell 5 - Honest test eval. Three layers:
#   A. Structural    -- valid JSON in the right shape (free, deterministic)
#   B. Grounded      -- output mentions the actual correct move + a known
#                       technique tag for THIS position (free, deterministic)
#   C. ManualJudge   -- writes 20 random samples to a queue file for you to
#                       grade by hand (you stamp 0..5 + a one-line reason)
# Headline metric is "USEFUL ANSWER %": JSON-valid + has correct comment +
# (mentions correct coord OR a known technique) + looks like English.
# This is what we should track across runs, not raw JSON-compliance.
# =====================================================================
import json, re, random, time

JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)
SGF_COORD_RE  = re.compile(r"\b[a-s]{2}\b")
TAGS_RE       = re.compile(r"^Tags: (.+)$",         re.MULTILINE)
BLACK_RE      = re.compile(r"^Black stones: (.+)$", re.MULTILINE)
WHITE_RE      = re.compile(r"^White stones: (.+)$", re.MULTILINE)
HINT_COORD_RE = re.compile(r"\{!([a-s]{2})\}")

# Subset of GO_TECHNIQUE words used for grounded matching
TECHNIQUE_WORDS = {
    "ladder", "snapback", "net", "throw-in", "throw", "tesuji", "atari",
    "ko", "seki", "eye", "sente", "gote", "shape", "shicho", "geta",
    "wedge", "hane", "cut", "connect", "capture", "kill", "live",
    "life", "death", "endgame", "joseki", "fuseki", "yose",
}

FastLanguageModel.for_inference(model)

def generate_one(messages, max_new_tokens=384):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_user(text):
    tags  = [t.strip() for t in (TAGS_RE.search(text).group(1).split(",")  if TAGS_RE.search(text)  else [])]
    black = [c.strip() for c in (BLACK_RE.search(text).group(1).split(",") if BLACK_RE.search(text) else [])]
    white = [c.strip() for c in (WHITE_RE.search(text).group(1).split(",") if WHITE_RE.search(text) else [])]
    return tags, black, white

def score_one(generated, reference, user_text):
    # Layer A
    s = {"parsed_ok": False, "has_correct": False, "has_wrong": False, "n_hints": 0,
         "n_chars_correct": 0, "n_chars_wrong": 0}
    blob = JSON_BLOCK_RE.search(generated or "")
    flat_text = generated or ""
    if blob:
        try:
            obj = json.loads(blob.group(0))
            s["parsed_ok"] = True
            tc = obj.get("teaching_comments") or {}
            cc = (tc.get("correct_comment") or "").strip() if isinstance(tc, dict) else ""
            wc = tc.get("wrong_comments") if isinstance(tc, dict) else None
            if isinstance(wc, dict):    wc_text = " ".join(str(v) for v in wc.values())
            elif isinstance(wc, list):  wc_text = " ".join(str(v) for v in wc)
            else:                       wc_text = ""
            s["has_correct"]    = bool(cc)
            s["has_wrong"]      = bool(wc_text.strip())
            s["n_chars_correct"] = len(cc)
            s["n_chars_wrong"]   = len(wc_text)
            hints = obj.get("hints") or []
            s["n_hints"] = len(hints) if isinstance(hints, list) else 0
            # Flatten everything for grounded checks
            parts = []
            def walk(x):
                if isinstance(x, str): parts.append(x)
                elif isinstance(x, dict):
                    for v in x.values(): walk(v)
                elif isinstance(x, list):
                    for v in x: walk(v)
            walk(obj)
            flat_text = " ".join(parts)
        except json.JSONDecodeError:
            pass

    # Layer B - position-anchored grounded checks
    tags, black, white = parse_user(user_text)
    correct_move = ""
    if reference:
        m = HINT_COORD_RE.search(reference)
        if m: correct_move = m.group(1)
    g = {"mentions_correct_move": False, "mentions_tag_technique": False,
         "no_off_board_coords": True, "looks_english": False,
         "correct_move_coord": correct_move}
    text_lower = flat_text.lower()
    if correct_move and correct_move in text_lower:
        g["mentions_correct_move"] = True
    tag_tokens = set()
    for t in tags:
        for piece in re.split(r"[-_,\s]+", t.lower()):
            if piece: tag_tokens.add(piece)
    interesting = tag_tokens & TECHNIQUE_WORDS
    if interesting:
        g["mentions_tag_technique"] = any(w in text_lower for w in interesting)
    elif tag_tokens:
        # Vacuous pass when puzzle has tags but none are recognisable techniques
        g["mentions_tag_technique"] = True
    else:
        g["mentions_tag_technique"] = True
    coord_set = {c.lower() for c in black + white}
    if correct_move: coord_set.add(correct_move)
    found = SGF_COORD_RE.findall(text_lower)
    unknown = [c for c in found if c not in coord_set]
    g["no_off_board_coords"] = len(unknown) <= max(1, len(found) // 3)
    letters = sum(1 for ch in flat_text if "a" <= ch.lower() <= "z")
    g["looks_english"] = (letters / max(len(flat_text), 1)) >= 0.6 if flat_text else False
    return s, g

print(f"Running inference on {len(test_rows)} test rows...")
results = []
manual_queue = []
t0 = time.time()
for i, ex in enumerate(test_rows):
    msgs = ex["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    reference   = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
    user_text   = next((m["content"] for m in msgs if m["role"] == "user"), "")
    generated   = generate_one(prompt_msgs)
    struct, ground = score_one(generated, reference, user_text)
    results.append({"idx": i, "structural": struct, "grounded": ground,
                    "generated_preview": generated[:600],
                    "reference_preview": (reference or "")[:600]})
    manual_queue.append({"idx": i, "user": user_text, "reference": reference,
                         "generated": generated, "score": None, "rationale": ""})
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(test_rows)}  ({(time.time()-t0)/60:.1f} min)")
print(f"Inference done in {(time.time()-t0)/60:.1f} min.")

# Aggregate
n = max(len(results), 1)
pct = lambda flag, key: round(100.0 * sum(1 for r in results if r[flag][key]) / n, 1)
useful = sum(
    1 for r in results
    if r["structural"]["parsed_ok"] and r["structural"]["has_correct"]
       and (r["grounded"]["mentions_correct_move"] or r["grounded"]["mentions_tag_technique"])
       and r["grounded"]["looks_english"]
)
summary = {
    "model": MODEL_NAME, "train_seconds": train_secs, "peak_vram_gb": peak_vram,
    "test_n": n, "elapsed_minutes": round((time.time()-t0)/60, 2),
    "useful_answer_pct": round(100.0 * useful / n, 1),
    "structural": {
        "json_compliance_pct": pct("structural", "parsed_ok"),
        "has_correct_pct":     pct("structural", "has_correct"),
        "has_wrong_pct":       pct("structural", "has_wrong"),
        "avg_hints":           round(sum(r["structural"]["n_hints"] for r in results) / n, 2),
        "avg_chars_correct":   round(sum(r["structural"]["n_chars_correct"] for r in results) / n, 1),
        "avg_chars_wrong":     round(sum(r["structural"]["n_chars_wrong"] for r in results) / n, 1),
    },
    "grounded": {
        "mentions_correct_move_pct":  pct("grounded", "mentions_correct_move"),
        "mentions_tag_technique_pct": pct("grounded", "mentions_tag_technique"),
        "no_off_board_coords_pct":    pct("grounded", "no_off_board_coords"),
        "looks_english_pct":          pct("grounded", "looks_english"),
    },
}
print("\n=== EVAL SUMMARY ===")
print(json.dumps(summary, indent=2))

# Layer C - sample 20 for manual review
random.seed(42)
manual_sample = random.sample(manual_queue, min(20, len(manual_queue)))
queue_path = os.path.join(OUT_DIR, "judge_queue.jsonl")
with open(queue_path, "w", encoding="utf-8") as f:
    for it in manual_sample:
        f.write(json.dumps(it, ensure_ascii=False) + "\n")
print(f"\n[Layer C] {len(manual_sample)} samples queued for manual review -> {queue_path}")
print("           Open the file, fill in `score` (0..5) and `rationale`, save.")

# Persist
with open(os.path.join(OUT_DIR, "test_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
with open(os.path.join(OUT_DIR, "test_results.json"), "w") as f:
    json.dump(results, f, indent=2)


## Save adapter (+ optional merged fp16) and download


In [ ]:
# =====================================================================
# Cell 6 - Save LoRA adapter, optional merged fp16, zip, download.
# =====================================================================
import shutil

adapter_dir = os.path.join(OUT_DIR, "adapter")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved LoRA adapter -> {adapter_dir}")

merged_dir = None
if SAVE_MERGED_FP16:
    merged_dir = os.path.join(OUT_DIR, "merged_fp16")
    try:
        model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
        print(f"Saved merged fp16 weights -> {merged_dir}")
    except Exception as e:
        print(f"WARNING: merged fp16 save failed ({type(e).__name__}: {e})")
        print("  Adapter-only save above is still usable; merge offline if needed.")
        merged_dir = None

# Zip everything for one-click download
artifact_root = os.path.join(OUT_DIR, "artifacts")
os.makedirs(artifact_root, exist_ok=True)
for fname in ("test_summary.json", "test_results.json", "judge_queue.jsonl"):
    src = os.path.join(OUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(artifact_root, fname))
shutil.copytree(adapter_dir, os.path.join(artifact_root, "adapter"), dirs_exist_ok=True)
if merged_dir:
    shutil.copytree(merged_dir, os.path.join(artifact_root, "merged_fp16"), dirs_exist_ok=True)

zip_base = os.path.join(OUT_DIR, "yen_sei_tier1_artifacts")
zip_path = shutil.make_archive(zip_base, "zip", artifact_root)
size_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f"\nZipped artifacts -> {zip_path}  ({size_mb:.1f} MB)")

# Optional HF push
if PUSH_TO_HF and HF_REPO_ID and HF_TOKEN:
    try:
        from huggingface_hub import HfApi  # type: ignore
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO_ID, exist_ok=True, private=True)
        api.upload_folder(folder_path=adapter_dir, repo_id=HF_REPO_ID, path_in_repo="adapter")
        if merged_dir:
            api.upload_folder(folder_path=merged_dir, repo_id=HF_REPO_ID, path_in_repo="merged_fp16")
        print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"HF push failed ({type(e).__name__}: {e})")

# Browser download (Colab only). Skipped automatically if not in Colab.
try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except ImportError:
    print(f"\nNot in Colab. Artifacts at: {artifact_root}")
